<a href="https://colab.research.google.com/github/sila85/production_ml_demo/blob/main/IMDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries if they are not already installed
%pip install -q tensorflow scikit-learn matplotlib seaborn

# Import libraries for data processing
import re
import random
import numpy as np
import pandas as pd
import html

# Import TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Import evaluation and visualization tools
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

In [ ]:
# Number of words to retain in the vocabulary
VOCAB_SIZE = 20000

# Maximum number of tokens per review
MAX_LENGTH = 250

# Embedding vector size
EMBEDDING_DIM = 128

# LSTM hidden units
LSTM_UNITS = 64

# Training settings
BATCH_SIZE = 64
EPOCHS = 8

print("Vocabulary size:", VOCAB_SIZE)
print("Maximum review length:", MAX_LENGTH)

In [ ]:
# Load the training and testing datasets
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE
)

# Display dataset information
print("Training reviews:", len(x_train))
print("Testing reviews:", len(x_test))

print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

print("First encoded review:")
print(x_train[0][:30])

print("First review label:", y_train[0])

In [ ]:
# Configuration parameters
VOCAB_SIZE = 20000  # Use the 20,000 most frequent words
MAX_LENGTH = 250    # Maximum number of tokens per review

# Load the dataset.
# num_words limits the vocabulary to the most frequent words.
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE
)

# Display dataset dimensions
print("Training reviews:", len(x_train))
print("Testing reviews:", len(x_test))
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

# Display a sample encoded review
print("\nEncoded review:")
print(x_train[0][:30])

# Display its sentiment label
print("Sentiment:", "Positive" if y_train[0] == 1 else "Negative")

In [ ]:
# Count positive and negative reviews
unique_labels, label_counts = np.unique(y_train, return_counts=True)

print("Training label distribution:")
for label, count in zip(unique_labels, label_counts):
    sentiment = "Positive" if label == 1 else "Negative"
    print(f"{sentiment}: {count}")

# Calculate review lengths
review_lengths = [len(review) for review in x_train]

print("\nShortest review:", min(review_lengths), "tokens")
print("Longest review:", max(review_lengths), "tokens")
print("Average review length:", round(np.mean(review_lengths), 2), "tokens")

# Plot the distribution of review lengths
plt.figure(figsize=(8, 5))
plt.hist(review_lengths, bins=50)
plt.xlabel("Number of tokens")
plt.ylabel("Number of reviews")
plt.title("Distribution of IMDB Review Lengths")
plt.show()

In [ ]:
# Obtain the mapping from words to integer IDs
word_index = imdb.get_word_index()

# Reverse the mapping: integer IDs to words
index_word = {index + 3: word for word, index in word_index.items()}

# Add special-token descriptions
index_word[0] = "<PAD>"
index_word[1] = "<START>"
index_word[2] = "<UNK>"
index_word[3] = "<UNUSED>"

def decode_review(encoded_review):
    """
    Convert a sequence of integer IDs into readable review text.
    """
    words = [index_word.get(token, "<UNK>") for token in encoded_review]
    return " ".join(words)

# Display one review
print("Decoded review:")
print(decode_review(x_train[0]))

print("\nSentiment label:", "Positive" if y_train[0] == 1 else "Negative")

In [ ]:
import html

In [ ]:
def clean_text(text):
    """
    Normalize readable review text.

    Steps:
    1. Decode HTML entities.
    2. Convert text to lowercase.
    3. Remove HTML tags.
    4. Remove unwanted characters.
    5. Normalize whitespace.
    """
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Demonstrate normalization on one review
sample_review = decode_review(x_train[0])

print("Before normalization:")
print(sample_review[:500])

print("\nAfter normalization:")
print(clean_text(sample_review)[:500])

In [ ]:
def clean_text(text):
    """
    Normalize readable review text.

    Steps:
    1. Decode HTML entities.
    2. Convert text to lowercase.
    3. Remove HTML tags.
    4. Remove unwanted characters.
    5. Normalize whitespace.
    """
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Demonstrate normalization on one review
sample_review = decode_review(x_train[0])

print("Before normalization:")
print(sample_review[:500])

print("\nAfter normalization:")
print(clean_text(sample_review)[:500])

In [ ]:
# Pad shorter reviews with zeros at the END.
# Truncate longer reviews at the END.
x_train_pad = pad_sequences(
    x_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

# Convert labels to integer arrays
y_train = np.asarray(y_train, dtype=np.int32)
y_test = np.asarray(y_test, dtype=np.int32)

print("Padded training shape:", x_train_pad.shape)
print("Padded testing shape:", x_test_pad.shape)

print("\nExample padded review:")
print(x_train_pad[0][:50])

In [ ]:
# Project settings
VOCAB_SIZE = 20000
MAX_LENGTH = 250
EMBEDDING_DIM = 128
LSTM_UNITS = 64
BATCH_SIZE = 64
EPOCHS = 8

print("Project settings defined successfully.")

In [ ]:
# Build the model
model = Sequential([
    # Convert token IDs into dense word vectors.
    # mask_zero=True tells the model to ignore padding tokens.
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True
    ),

    # Read the review in both directions
    Bidirectional(
        LSTM(LSTM_UNITS)
    ),

    # Reduce overfitting
    Dropout(0.5),

    # Output probability between 0 and 1
    Dense(1, activation="sigmoid")
])

# Configure model training
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display the architecture
model.summary()

In [ ]:
# Stop training when validation loss stops improving.
# Restore the best model weights.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    x_train_pad,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Plot accuracy across epochs
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot loss across epochs
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Evaluate the model on unseen test reviews
test_loss, test_accuracy = model.evaluate(
    x_test_pad,
    y_test,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy (%): {test_accuracy * 100:.2f}%")

In [ ]:
# Predict sentiment probabilities for the test set
y_prob = model.predict(x_test_pad, batch_size=BATCH_SIZE, verbose=1)

# Convert probabilities to binary labels using a 0.5 threshold
y_pred = (y_prob >= 0.5).astype(int).ravel()

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("TEST PERFORMANCE")
print("----------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

# Detailed report for both sentiment classes
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"],
        zero_division=0
    )
)

# Plot the confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
def encode_new_review(review_text):
    """
    Convert a new review into IMDB-compatible token IDs.

    IMDB word IDs are offset by 3 because IDs 0-3
    are reserved for special tokens.
    """
    # Basic normalization
    review_text = clean_text(review_text)

    # Split text into words
    words = review_text.split()

    # Convert words into IDs
    encoded = []

    for word in words:
        original_id = word_index.get(word)

        # IDs 0-3 are reserved; unknown words use ID 2.
        if original_id is None:
            token_id = 2
        else:
            token_id = original_id + 3

            # Treat words outside the model's vocabulary as unknown.
            if token_id >= VOCAB_SIZE:
                token_id = 2

        encoded.append(token_id)

    # Keep the input within the model's vocabulary range
    return encoded


def predict_sentiment(review_text):
    """
    Predict whether a review is positive or negative.
    """
    # Encode the review
    encoded_review = encode_new_review(review_text)

    # Pad the review using the same settings as training
    padded_review = pad_sequences(
        [encoded_review],
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post",
        value=0
    )

    # Predict positive sentiment probability
    probability = float(model.predict(padded_review, verbose=0)[0][0])

    # Classify using a 0.5 threshold
    sentiment = "Positive" if probability >= 0.5 else "Negative"

    return sentiment, probability

In [ ]:
# Example reviews for testing
sample_reviews = [
    "This movie was fantastic. The acting was excellent and the story was moving.",
    "I hated this film. It was boring, poorly written, and a complete waste of time.",
    "The movie was enjoyable and the actors performed very well.",
    "The plot was confusing and the movie was disappointing."
]

# Run predictions
for review in sample_reviews:
    sentiment, probability = predict_sentiment(review)

    print("Review:", review)
    print("Predicted sentiment:", sentiment)
    print(f"Positive probability: {probability:.4f}")
    print("-" * 70)

In [ ]:
# Save the trained neural network
model.save("imdb_bilstm_sentiment.keras")

print("Model saved successfully.")